# Entity Reuse Processing


In [1]:
import pandas as pd
import numpy as np
import pyarrow
from pathlib import Path
from collections import defaultdict, deque
from tqdm import tqdm
from datetime import datetime
import json

In [ ]:
# Parameters
folder_output_path = Path("../experimental_results/Labeled_Datasets/Ecuador/Ecuador_part_all/2026_01_10")
df_path = folder_output_path / "processed_Ecuador_part_all.gzip.parquet"

In [3]:
output_df_path = df_path
# output_df_path = folder_output_path / "processed_Ecuador_part_all_v2.gzip.parquet"

print(f"Output df path: {output_df_path}")

Output df path: ../results/Labeled_Datasets/Ecuador/Ecuador_part_all/2026_01_10/processed_Ecuador_part_all.gzip.parquet


In [ ]:
# Load data
df = pd.read_parquet(df_path)
print(f"Loaded df from: {df_path}")
print(f"Loaded df with shape: {df.shape}")

# for test
# df= df.sample(n=10**4)
# print(f"TEST df with shape: {df.shape}")

In [5]:
# write_report

REPORT_PATH = folder_output_path / "progress_report.txt"

def write_report(msg: str):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"{timestamp} | {msg}"
    print(line)
    with open(REPORT_PATH, "a") as f:
        f.write(line + "\n")

write_report(f"Started entity reuse processing for {df_path}")

2026-03-03 11:01:42 | Started entity reuse processing for ../results/Labeled_Datasets/Ecuador/Ecuador_part_all/2026_01_10/processed_Ecuador_part_all.gzip.parquet


In [ ]:
def add_entity_reuse(
    df: pd.DataFrame,
    time_window: str = "4D",
    entity_column: str = "hashtags",
    post_id_column: str = "postid",
    account_id_column: str = "accountid",
    timestamp_column: str = "post_time",
) -> None:
    """
    Adds temporal entity-reuse columns to a DataFrame.

    For each post, this function finds earlier posts within a `time_window` that share at least one entity.
    It adds columns with the post IDs and account IDs of these related posts.

    It maintains per-entity sliding windows to efficiently track recent usage,
    collects matching prior posts, and stores their post IDs and account IDs in new columns.
    It use dict{entity: deque[(timestamp, post_index)]} to track recent posts for each entity, expiring old entries as it goes.

    Note 1: An authors can reuse entities from their own earlier posts, and these will be included in the results. You can later set to ignore self-reuse when building the graph.
    Note 2: The function does not count for a post that reuses the same entity type multiple times between posts. It will only be counted once per post.

    Args:
        df (pd.DataFrame): The input DataFrame.
        entity_column (str): The column with entities for each post.
        time_window (str): The sliding window duration (default: "4D").
        post_id_column (str): The column with post identifiers (default: "postid").
        account_id_column (str): The column with account identifiers (default: "accountid").
        timestamp_column (str): The column with post timestamps (default: "post_time").

    Returns:
        None: Modifies the DataFrame in-place, adding:
              - f"{entity_column}_reuse_{time_window}_postids"
              - f"{entity_column}_reuse_{time_window}_accountids"
    """

    write_report(f"\n[add_entity_reuse] Adding entity reuse columns for '{entity_column}' with time window {time_window}...")

    # 1. Define output column names
    output_post_ids_column = f"{entity_column}_reuse_{time_window}_postids"
    output_account_ids_column  = f"{entity_column}_reuse_{time_window}_accountids"
    
    # Drop existing columns if they already exist to avoid conflicts
    existing_cols_to_drop = [col for col in [output_post_ids_column, output_account_ids_column] if col in df.columns]
    if existing_cols_to_drop:
        write_report(f"Dropping existing columns: {existing_cols_to_drop}")
        df.drop(columns=existing_cols_to_drop, inplace=True)

    # 2. Convert columns to NumPy arrays for performance
    timestamps = pd.to_datetime(df[timestamp_column]).to_numpy(dtype="datetime64[ns]")
    post_ids = df[post_id_column].to_numpy()
    account_ids = df[account_id_column].to_numpy()
    entities_per_post = df[entity_column].to_numpy()

    # Convert time window to numpy timedelta
    time_window_delta = np.timedelta64(pd.Timedelta(time_window).value, "ns")

    # Pre-allocate output lists
    num_posts = len(df)
    output_post_ids = [[] for _ in range(num_posts)]
    output_account_ids  = [[] for _ in range(num_posts)]

    # 3. Set up entity history tracker: {entity: deque[(timestamp, post_index)]}
    entities_history: dict[str, deque] = defaultdict(deque)

    # 4. Process posts in chronological order
    sorted_timestamp_indices = np.argsort(timestamps)
    for current_post_index in tqdm(sorted_timestamp_indices, desc="Processing posts"):
        current_timestamp = timestamps[current_post_index]
        current_entities = entities_per_post[current_post_index]

        # no entities in this post
        if current_entities is None or (len(current_entities) == 0):
            continue

        matching_post_indices = set()

        # 5. Find related earlier posts for each entity
        for entity in current_entities:
            
            # no entity
            if not entity:
                continue

            entity_post_deque = entities_history[entity] # deque: [(timestamp, post_index)] of post with this entity
            
            # Expire old posts from the deque
            while entity_post_deque and (current_timestamp - entity_post_deque[0][0]) > time_window_delta:
                entity_post_deque.popleft() # remove posts outside the time window

            # Collect indices of posts still in the window
            matching_post_indices.update(post_index for _, post_index in entity_post_deque)

        # 6. Store results in df
        if matching_post_indices:
            sorted_matching_indices = sorted(list(matching_post_indices), key=lambda index: timestamps[index]) # label order by timestamps, oldest is first

            # Remove duplicates while preserving order
            output_post_ids[current_post_index] = list(dict.fromkeys([post_ids[post_index] for post_index in sorted_matching_indices]))
            output_account_ids[current_post_index] = list(dict.fromkeys([account_ids[post_index] for post_index in sorted_matching_indices]))

        # 7. Add current post to history (after matching)
        for entity in current_entities:
            if entity:
                entities_history[entity].append((current_timestamp, current_post_index))

    # 8. Convert outputs to lists and add to df
    # Had issues since this column contains long lists. Parquet ran into OOM with gzip compression and ArrowNotImplementedError when reading it back.
    # df[output_post_ids_column] = [json.dumps(x) for x in output_post_ids]
    # print(f"[add_entity_reuse] Added column: {output_post_ids_column}")

    df[output_account_ids_column] = [list(x) for x in output_account_ids]
    print(f"[add_entity_reuse] Added column: {output_account_ids_column}")

In [ ]:
# Configuration for entity reuse
ENTITY_COLS = ["hashtags", "urls", "account_mentions", "ner_entities"]
time_windows = ["12H", "3D"] # See pandas.TimeDelta documentation. Example: "1W" for 1 week, "4D" for 4 days, "12H" for 12 hours, etc.


# #  for test
# ENTITY_COLS = ["hashtags"]
# time_windows = ["4D"]

In [ ]:
# Execute entity reuse processing
for time_window in time_windows:
    for entity_col in ENTITY_COLS:
        add_entity_reuse(df, time_window=time_window, entity_column=entity_col)

In [ ]:
# Save results
df.to_parquet(output_df_path, index=False, compression="gzip")
write_report(f"Saved results to: {output_df_path}")
write_report(f"✓ Entity reuse processing complete!")

# Test Reading from Disk

In [8]:
import pyarrow.parquet as pq

pf = pq.ParquetFile(output_df_path)

for col in pf.schema_arrow.names:  # top-level columns only
    try:
        write_report(f"Converting: {col}")
        t = pf.read(columns=[col])     # arrow table
        _ = t.to_pandas()              # <-- this is where it often fails
        write_report("  OK")
    except Exception as e:
        write_report(f"  ❌ FAIL column: {col}")
        write_report(f"  Error: {repr(e)}")

2026-03-03 11:02:21 | Converting: postid
2026-03-03 11:02:22 |   OK
2026-03-03 11:02:22 | Converting: post_text
2026-03-03 11:02:24 |   OK
2026-03-03 11:02:24 | Converting: application_name
2026-03-03 11:02:24 |   OK
2026-03-03 11:02:24 | Converting: post_language
2026-03-03 11:02:24 |   OK
2026-03-03 11:02:24 | Converting: in_reply_to_postid
2026-03-03 11:02:24 |   OK
2026-03-03 11:02:24 | Converting: in_reply_to_accountid
2026-03-03 11:02:24 |   OK
2026-03-03 11:02:24 | Converting: post_time
2026-03-03 11:02:24 |   OK
2026-03-03 11:02:24 | Converting: accountid
2026-03-03 11:02:24 |   OK
2026-03-03 11:02:24 | Converting: account_profile_description
2026-03-03 11:02:25 |   OK
2026-03-03 11:02:25 | Converting: follower_count
2026-03-03 11:02:25 |   OK
2026-03-03 11:02:25 | Converting: following_count
2026-03-03 11:02:25 |   OK
2026-03-03 11:02:25 | Converting: account_creation_date
2026-03-03 11:02:25 |   OK
2026-03-03 11:02:25 | Converting: is_repost
2026-03-03 11:02:25 |   OK
2026-03

In [9]:
df = pd.read_parquet(output_df_path)
write_report(f"Re-loaded df from: {output_df_path}")
write_report(f"Re-loaded df with shape: {df.shape}")

2026-03-03 11:03:20 | Re-loaded df from: ../results/Labeled_Datasets/Ecuador/Ecuador_part_all/2026_01_10/processed_Ecuador_part_all.gzip.parquet
2026-03-03 11:03:20 | Re-loaded df with shape: (1468565, 37)


In [10]:
write_report(df.dtypes)

2026-03-03 11:03:28 | postid                                              str
post_text                                           str
application_name                                    str
post_language                                       str
in_reply_to_postid                                  str
in_reply_to_accountid                               str
post_time                                datetime64[us]
accountid                                           str
account_profile_description                         str
follower_count                                    int64
following_count                                   int64
account_creation_date                            object
is_repost                                          bool
reposted_accountid                                  str
reposted_postid                                     str
hashtags                                         object
urls                                             object
account_mentions          

In [11]:
df.head()

,postid,post_text,application_name,post_language,in_reply_to_postid,in_reply_to_accountid,post_time,accountid,account_profile_description,follower_count,...,BERTopic_prob_128,ner_entities,hashtags_reuse_12H_accountids,urls_reuse_12H_accountids,account_mentions_reuse_12H_accountids,ner_entities_reuse_12H_accountids,hashtags_reuse_3D_accountids,urls_reuse_3D_accountids,account_mentions_reuse_3D_accountids,ner_entities_reuse_3D_accountids
0,0bafe020a260495bee38b98fde83668d924454019b,Vamos a cambiar la forma de hacer comunicación...,2727201c363b6fc4afe259ba4094ae0c49cb4df224,spa_Latn,NaN,NaN,2010-03-13 02:52:00,22de67d991b31493165b9270f7d8b5b4ce712d5fc8,"#Periodismo, #RelacionesPúblicas, #Comunicac...",1528,...,0.694254,[],[],[],[],[],[],[],[],[]
1,608fba51f2460c7f1ca527dfc48de2bc858d4ad201,"Damos servicio de manejo de prensa, elaboració...",2727201c363b6fc4afe259ba4094ae0c49cb4df224,spa_Latn,NaN,NaN,2010-03-13 03:08:00,22de67d991b31493165b9270f7d8b5b4ce712d5fc8,"#Periodismo, #RelacionesPúblicas, #Comunicac...",1528,...,0.756225,[],[],[],[],[],[],[],[],[]
2,514b2c38b26e240cac85382be74ed7d61a02468ac1,@db7c47cef2aa2ba68b5c010d761a03fc1c0f0d1302 Bu...,2727201c363b6fc4afe259ba4094ae0c49cb4df224,spa_Latn,f29b32f8dbaa1e0f00152decbcc2bd93dddc740464,db7c47cef2aa2ba68b5c010d761a03fc1c0f0d1302,2010-03-13 15:26:00,2c6c0ef5f445760177b016e33b36809bde3c7ac241,NaN,35,...,0.694943,"[well@@, masks]",[],[],[],[],[],[],[],[]
3,e82a816068e88387d19430348f0420b040e0d11b4e,"Mario, debemos denuciar que en los poderes pub...",2727201c363b6fc4afe259ba4094ae0c49cb4df224,spa_Latn,NaN,NaN,2010-03-13 15:32:00,2c6c0ef5f445760177b016e33b36809bde3c7ac241,NaN,35,...,0.801221,"[mario@@, chavista]",[],[],[],[],[],[],[],[]
4,13ed6aaeec4fffaf840f1c42ff88f5ab2cb324be64,Tambien debemos estar claro que la oposición e...,2727201c363b6fc4afe259ba4094ae0c49cb4df224,spa_Latn,NaN,NaN,2010-03-13 15:35:00,2c6c0ef5f445760177b016e33b36809bde3c7ac241,NaN,35,...,0.673560,[],[],[],[],[],[],[],[],[]
